In [1]:
from abc import ABC, abstractmethod
from functools import total_ordering
import math


def _check_positive(value, name):
    """Проверяет, что значение положительное."""
    if value <= 0:
        raise ValueError(f"значение {name} должно быть положительным")
    return value


def _check_scale(k):
    """Проверяет коэффициент масштабирования."""
    if not isinstance(k, (int, float)):
        return False
    if k <= 0:
        raise ValueError("коэффициент масштабирования должен быть положительным")
    return True


@total_ordering
class Shape(ABC):
    """Абстрактная плоская фигура."""

    @abstractmethod
    def area(self):
        """Возвращает площадь фигуры."""

    @abstractmethod
    def perimeter(self):
        """Возвращает периметр фигуры."""

    def describe(self):
        """Строковое описание фигуры."""
        return (
            f"{type(self).__name__}: "
            f"площадь {self.area():.2f}, периметр {self.perimeter():.2f}"
        )

    def __str__(self):
        """Представление для пользователя."""
        return f"{type(self).__name__} площадь {self.area():.2f}"

    def __eq__(self, other):
        """Равенство фигур по площади с допуском."""
        if not isinstance(other, Shape):
            return NotImplemented
        return math.isclose(
            self.area(), other.area(), rel_tol=1e-9, abs_tol=1e-9
        )

    def __lt__(self, other):
        """Сравнение фигур по площади."""
        if not isinstance(other, Shape):
            return NotImplemented
        return self.area() < other.area()

    def __hash__(self):
        """Хеш согласован с равенством по площади."""
        return hash(round(self.area(), 9))


class Circle(Shape):
    """Круг."""

    def __init__(self, r):
        """Создаёт круг радиуса r."""
        self.r = r

    @property
    def r(self):
        """Радиус круга."""
        return self._r

    @r.setter
    def r(self, value):
        """Устанавливает положительный радиус."""
        self._r = _check_positive(value, "радиус")

    def area(self):
        """Площадь круга."""
        return math.pi * self._r ** 2

    def perimeter(self):
        """Длина окружности."""
        return 2 * math.pi * self._r

    def __repr__(self):
        """Представление для программиста."""
        return f"Circle(r={self._r:g})"

    def __mul__(self, k):
        """Масштабирование круга."""
        if not _check_scale(k):
            return NotImplemented
        return Circle(self._r * k)


class Rectangle(Shape):
    """Прямоугольник."""

    def __init__(self, width, height):
        """Создаёт прямоугольник с заданными сторонами."""
        self.width = width
        self.height = height

    @property
    def width(self):
        """Ширина прямоугольника."""
        return self._width

    @width.setter
    def width(self, value):
        """Устанавливает положительную ширину."""
        self._width = _check_positive(value, "ширина")

    @property
    def height(self):
        """Высота прямоугольника."""
        return self._height

    @height.setter
    def height(self, value):
        """Устанавливает положительную высоту."""
        self._height = _check_positive(value, "высота")

    def area(self):
        """Площадь прямоугольника."""
        return self._width * self._height

    def perimeter(self):
        """Периметр прямоугольника."""
        return 2 * (self._width + self._height)

    def __repr__(self):
        """Представление для программиста."""
        return f"Rectangle({self._width:g}, {self._height:g})"

    def __mul__(self, k):
        """Масштабирование прямоугольника."""
        if not _check_scale(k):
            return NotImplemented
        return Rectangle(self._width * k, self._height * k)


class Triangle(Shape):
    """Треугольник."""

    def __init__(self, a, b, c):
        """Создаёт треугольник со сторонами a, b, c."""
        self._validate_sides(a, b, c)
        self._a = a
        self._b = b
        self._c = c

    @staticmethod
    def _validate_sides(a, b, c):
        """Проверяет положительность сторон и неравенство треугольника."""
        for name, value in (("a", a), ("b", b), ("c", c)):
            if value <= 0:
                raise ValueError(f"сторона {name} должна быть положительной")
        if not (a + b > c and a + c > b and b + c > a):
            raise ValueError(
                f"стороны {a:g}, {b:g}, {c:g} не образуют треугольник"
            )

    @property
    def a(self):
        """Сторона a."""
        return self._a

    @a.setter
    def a(self, value):
        """Устанавливает сторону a с проверкой."""
        self._validate_sides(value, self._b, self._c)
        self._a = value

    @property
    def b(self):
        """Сторона b."""
        return self._b

    @b.setter
    def b(self, value):
        """Устанавливает сторону b с проверкой."""
        self._validate_sides(self._a, value, self._c)
        self._b = value

    @property
    def c(self):
        """Сторона c."""
        return self._c

    @c.setter
    def c(self, value):
        """Устанавливает сторону c с проверкой."""
        self._validate_sides(self._a, self._b, value)
        self._c = value

    def area(self):
        """Площадь треугольника по формуле Герона."""
        p = (self._a + self._b + self._c) / 2
        return math.sqrt(
            max(0.0, p * (p - self._a) * (p - self._b) * (p - self._c))
        )

    def perimeter(self):
        """Периметр треугольника."""
        return self._a + self._b + self._c

    def __repr__(self):
        """Представление для программиста."""
        return f"Triangle({self._a:g}, {self._b:g}, {self._c:g})"

    def __mul__(self, k):
        """Масштабирование треугольника."""
        if not _check_scale(k):
            return NotImplemented
        return Triangle(self._a * k, self._b * k, self._c * k)


class Square(Rectangle):
    """Квадрат — прямоугольник с равными сторонами."""

    def __init__(self, side):
        """Создаёт квадрат со стороной side."""
        super().__init__(side, side)

    @property
    def side(self):
        """Сторона квадрата."""
        return self._width

    @side.setter
    def side(self, value):
        """Устанавливает сторону квадрата, сохраняя равенство сторон."""
        value = _check_positive(value, "сторона квадрата")
        self._width = value
        self._height = value

    @property
    def width(self):
        """Ширина квадрата."""
        return self._width

    @width.setter
    def width(self, value):
        """Изменение ширины квадрата меняет и высоту."""
        self.side = value

    @property
    def height(self):
        """Высота квадрата."""
        return self._height

    @height.setter
    def height(self, value):
        """Изменение высоты квадрата меняет и ширину."""
        self.side = value

    def __repr__(self):
        """Представление для программиста."""
        return f"Square(a={self.side:g})"

    def __mul__(self, k):
        """Масштабирование квадрата."""
        if not _check_scale(k):
            return NotImplemented
        return Square(self.side * k)


class Drawing:
    """Чертёж — контейнер фигур."""

    def __init__(self, name):
        """Создаёт чертёж с названием."""
        self.name = name
        self._shapes = []

    def add(self, shape):
        """Добавляет фигуру в чертёж."""
        if not isinstance(shape, Shape):
            raise TypeError("в чертёж можно добавлять только объекты Shape")
        self._shapes.append(shape)

    def __len__(self):
        """Число фигур на чертеже."""
        return len(self._shapes)

    def __contains__(self, shape):
        """Проверяет наличие фигуры на чертеже."""
        return shape in self._shapes

    def __getitem__(self, index):
        """Доступ к фигуре по индексу."""
        return self._shapes[index]

    def __iter__(self):
        """Перебор фигур на чертеже."""
        return iter(self._shapes)

    def total_area(self):
        """Суммарная площадь всех фигур."""
        return sum(shape.area() for shape in self._shapes)

    def total_perimeter(self):
        """Суммарный периметр всех фигур."""
        return sum(shape.perimeter() for shape in self._shapes)

    def largest(self, n):
        """Возвращает n крупнейших фигур по площади."""
        return sorted(
            self._shapes, key=lambda shape: shape.area(), reverse=True
        )[:n]

    def __str__(self):
        """Представление чертежа для пользователя."""
        lines = [f"Чертёж «{self.name}»: {len(self)} фигур"]
        for shape in sorted(self._shapes):
            lines.append(
                f"  {shape!r:30} "
                f"площадь {shape.area():7.2f}  "
                f"периметр {shape.perimeter():7.2f}"
            )
        return "\n".join(lines)

    def __repr__(self):
        """Представление чертежа для программиста."""
        return f"Drawing({self.name!r}, {len(self)} фигур)"


# --- Демонстрация ---

drawing = Drawing("План участка")

shapes = [
    Circle(2),
    Rectangle(3, 4),
    Triangle(3, 4, 5),
    Square(3),
    Circle(1.5),
    Rectangle(2, 5),
    Triangle(6, 8, 10),
    Square(4),
    Circle(0.5),
]

for shape in shapes:
    drawing.add(shape)

print(drawing)
print(f"\nСуммарная площадь: {drawing.total_area():.2f}")
print(f"Суммарный периметр: {drawing.total_perimeter():.2f}")

print("\nКрупнейшие фигуры:")
for shape in drawing.largest(3):
    print(" ", shape.describe())

print("\nSquare(a=3) == Rectangle(3, 3):", Square(3) == Rectangle(3, 3))

print("\nПроверка ошибок:")
for action in (
    lambda: Circle(-1),
    lambda: Triangle(1, 2, 10),
    lambda: Square(0),
):
    try:
        action()
    except ValueError as error:
        print("Ошибка:", error)

print("\nМасштабирование:")
base_circle = Circle(2)
scaled_circle = base_circle * 2
print(base_circle, "->", scaled_circle)
print(
    f"Площадь выросла в "
    f"{scaled_circle.area() / base_circle.area():.2f} раза"
)

base_square = Square(3)
scaled_square = base_square * 2
print(base_square, "->", scaled_square)
print(
    f"Площадь выросла в "
    f"{scaled_square.area() / base_square.area():.2f} раза"
)

Чертёж «План участка»: 9 фигур
  Circle(r=0.5)                  площадь    0.79  периметр    3.14
  Triangle(3, 4, 5)              площадь    6.00  периметр   12.00
  Circle(r=1.5)                  площадь    7.07  периметр    9.42
  Square(a=3)                    площадь    9.00  периметр   12.00
  Rectangle(2, 5)                площадь   10.00  периметр   14.00
  Rectangle(3, 4)                площадь   12.00  периметр   14.00
  Circle(r=2)                    площадь   12.57  периметр   12.57
  Square(a=4)                    площадь   16.00  периметр   16.00
  Triangle(6, 8, 10)             площадь   24.00  периметр   24.00

Суммарная площадь: 97.42
Суммарный периметр: 117.13

Крупнейшие фигуры:
  Triangle: площадь 24.00, периметр 24.00
  Square: площадь 16.00, периметр 16.00
  Circle: площадь 12.57, периметр 12.57

Square(a=3) == Rectangle(3, 3): True

Проверка ошибок:
Ошибка: значение радиус должно быть положительным
Ошибка: стороны 1, 2, 10 не образуют треугольник
Ошибка: значение